### This notebook updates the min and max values used for normalization of the risk indicator values in the BD_ClimateRisk_IKI sqlite database.

**Created:** 1/26/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 1/26/2025 by Sophia Bakar
 
**Status:** Complete
 
**Notes:** This script queries the IKI Climate Risk SQL database to get the max and min indicator values from the WaterALLOC indicators, Dynamic Indicators, and Static indicators, and updates the min/max values that are stored in the Indicators table. We buffer the maximum by 5% to account for significant differences in the max values of future scenarios or the addition of more basins. 

In [1]:
import pandas as pd
import sqlite3
import numpy as np

In [2]:
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Externo\PACA_Peru\Indicadores\BD_RiesgoClimatico_IKI.db"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [3]:
# Tables that contain indicator values
value_tables = [
    "IndValues_Dyn",
    "IndValues_Static",
    "IndValues_WaALLOC"
]

In [4]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

indid_sets = []

for tbl in value_tables:
    df = pd.read_sql_query(
        f"SELECT DISTINCT IndID FROM {tbl} WHERE Value IS NOT NULL;",
        conn
    )
    indid_sets.append(set(df["IndID"]))

all_indids = sorted(set.union(*indid_sets))

print(f"Found {len(all_indids)} indicators with values\n")

for ind_id in all_indids:

    ind_row = pd.read_sql_query(
        """
        SELECT Min, Max
        FROM Indicators
        WHERE IndID = ?
        """,
        conn,
        params=(ind_id,)
    )

    if ind_row.empty:
        print(f"⚠️ IndID {ind_id} not found in Indicators — skipping")
        continue

    old_min = ind_row.loc[0, "Min"]
    old_max = ind_row.loc[0, "Max"]

    values = []

    for tbl in value_tables:
        vdf = pd.read_sql_query(
            f"""
            SELECT Value
            FROM {tbl}
            WHERE IndID = ? AND Value IS NOT NULL
            """,
            conn,
            params=(ind_id,)
        )

        if not vdf.empty:
            values.extend(vdf["Value"].tolist())

    if not values:
        print(f"⚠️ IndID {ind_id}: no values found — skipping")
        continue

    values_arr = np.array(values, dtype=float)

    raw_min = values_arr.min()
    raw_max = values_arr.max()

    # updated min logic:
    # if raw_min is negative:
    #   1) use 0 if 0 exists in the values
    #   2) otherwise use the smallest positive nonzero value
    #   3) if no positive value exists, fall back to 0
    if raw_min < 0:
        if np.any(values_arr == 0):
            new_min = 0
        else:
            positive_vals = values_arr[values_arr > 0]

            if len(positive_vals) > 0:
                new_min = positive_vals.min()
            else:
                new_min = 0
    else:
        new_min = raw_min

    close_to_1_threshold = 1.05   # raw_max <= 1.05 → keep max=1
    expand_multiplier = 1.05      # buffer for expanded max

    if old_max == 1:
        if raw_max <= close_to_1_threshold:
            new_max = 1
        else:
            new_max = raw_max * expand_multiplier

    elif old_max == 4:
        # keep 4 fixed
        new_max = 4

    else:
        new_max = raw_max * expand_multiplier

    cursor.execute(
        """
        UPDATE Indicators
        SET Min = ?, Max = ?
        WHERE IndID = ?
        """,
        (float(new_min), float(new_max), ind_id)
    )

    print(
        f"IndID {ind_id}: "
        f"Min {old_min} → {new_min}, "
        f"Max {old_max} → {new_max} "
        f"(raw min = {raw_min}, raw max = {raw_max})"
    )

conn.commit()
conn.close()

print("\n✅ Indicator Min/Max update complete.")

Found 52 indicators with values

IndID 101: Min 1 → 1.0, Max 4 → 4 (raw min = 1.0, raw max = 4.0)
IndID 102: Min 1 → 1.0, Max 4 → 4 (raw min = 1.0, raw max = 4.0)
IndID 103: Min 0.23089489200632785 → 0.23089489200632785, Max 20244.30667973661 → 20244.30667973661 (raw min = 0.23089489200632785, raw max = 19280.292075939626)
IndID 104: Min 1 → 1.0, Max 4 → 4 (raw min = 1.0, raw max = 4.0)
IndID 105: Min 0 → 0.0, Max 50.2845 → 50.2845 (raw min = 0.0, raw max = 47.89)
IndID 106: Min 1 → 1.0, Max 4 → 4 (raw min = 1.0, raw max = 4.0)
IndID 107: Min 1 → 1.0, Max 4 → 4 (raw min = 1.0, raw max = 4.0)
IndID 108: Min 0 → 0.0, Max 1494.9301130536776 → 1494.9301130536776 (raw min = 0.0, raw max = 1423.7429648130262)
IndID 110: Min 0.25 → 0.25, Max 1 → 1 (raw min = 0.25, raw max = 1.0)
IndID 111: Min 0 → 0.0, Max 1 → 1 (raw min = 0.0, raw max = 0.75)
IndID 112: Min 0.25 → 0.25, Max 1 → 1 (raw min = 0.25, raw max = 1.0)
IndID 113: Min 0 → 0.0, Max 6675.423199289662 → 6675.423199289662 (raw min = 0.0,